In [1]:
!pip install google-generativeai

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import google.generativeai as genai


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
load_dotenv(override=True)
api_key = os.getenv("GEM_API_KEY")
genai.configure(api_key=api_key)

In [3]:
message = "Hello, Gemini! This is my first ever message to you! Hi!"
model = genai.GenerativeModel("gemini-2.5-flash-lite")
response = model.generate_content(message)
print(response.text)

Hi there! It's great to hear from you, and welcome! I'm really happy to be your first message. 😊

How can I help you today? Do you have any questions, or is there anything you'd like to talk about? I'm here and ready to chat!


In [4]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A class to represent and extract readable content from a given website.

    This class:
        - Fetches the HTML content of a webpage from a given URL.
        - Parses the HTML using BeautifulSoup.
        - Extracts the webpage title (if available).
        - Removes non-readable elements such as <script>, <style>, <img>, and <input>.
        - Extracts and stores cleaned, visible text from the page body.

    Attributes:
        url (str): The URL of the website.
        title (str): The extracted title of the webpage or a fallback string.
        text (str): The visible text content from the webpage body after cleaning.
    """

    def __init__(self, url):
        """
        Initialize a Website object by scraping and processing its content.

        Args:
            url (str): The full URL of the website to scrape.

        Raises:
            requests.exceptions.RequestException: If there is a network issue or the URL is invalid.
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = "No body content found."


In [5]:
web = Website("https://www.bbc.com/news")
print(web.title)
print(web.text)

BBC News - Breaking news, video and the latest top stories from the U.S. and around the world
Skip to content
British Broadcasting Corporation
Home
News
Sport
Business
Innovation
Culture
Arts
Travel
Earth
Audio
Video
Live
Israel-Gaza War
War in Ukraine
US & Canada
UK
Africa
Asia
Australia
Europe
Latin America
Middle East
In Pictures
BBC InDepth
BBC Verify
Home
News
Israel-Gaza War
War in Ukraine
US & Canada
UK
UK Politics
England
N. Ireland
N. Ireland Politics
Scotland
Scotland Politics
Wales
Wales Politics
Africa
Asia
China
India
Australia
Europe
Latin America
Middle East
In Pictures
BBC InDepth
BBC Verify
Sport
Business
Executive Lounge
Technology of Business
Future of Business
Innovation
Technology
Science & Health
Artificial Intelligence
AI v the Mind
Culture
Film & TV
Music
Art & Design
Style
Books
Entertainment News
Arts
Arts in Motion
Travel
Destinations
Africa
Antarctica
Asia
Australia and Pacific
Caribbean & Bermuda
Central America
Europe
Middle East
North America
South Americ

In [6]:
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related."

In [7]:
def user_prompt(website):
    """
    Generate a text prompt containing a website's title and content for summarization.

    This function constructs a prompt string that includes:
      - The website's title.
      - Instructions requesting a short summary of the website content.
      - The full text content of the website.
    If the website contains news or announcements, the instructions ask
    to summarize those as well.

    Args:
        website (Website): An instance of the Website class with `title` and `text` attributes.

    Returns:
        str: A formatted prompt string ready to be used for summarization by an AI model.
    """
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt


In [8]:
print(user_prompt(web))

You are looking at a website titled BBC News - Breaking news, video and the latest top stories from the U.S. and around the world
The contents of this website is as follows; please provide a short summary of this website. If it includes news or announcements, then summarize these too.

Skip to content
British Broadcasting Corporation
Home
News
Sport
Business
Innovation
Culture
Arts
Travel
Earth
Audio
Video
Live
Israel-Gaza War
War in Ukraine
US & Canada
UK
Africa
Asia
Australia
Europe
Latin America
Middle East
In Pictures
BBC InDepth
BBC Verify
Home
News
Israel-Gaza War
War in Ukraine
US & Canada
UK
UK Politics
England
N. Ireland
N. Ireland Politics
Scotland
Scotland Politics
Wales
Wales Politics
Africa
Asia
China
India
Australia
Europe
Latin America
Middle East
In Pictures
BBC InDepth
BBC Verify
Sport
Business
Executive Lounge
Technology of Business
Future of Business
Innovation
Technology
Science & Health
Artificial Intelligence
AI v the Mind
Culture
Film & TV
Music
Art & Design
Styl

In [9]:
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

In [10]:
gemini_messages = []
for m in messages:
    role = m["role"]
    if role == "system":
        # Merge into next user message if possible
        if gemini_messages and gemini_messages[-1]["role"] == "user":
            gemini_messages[-1]["parts"][0] = m["content"] + "\n\n" + gemini_messages[-1]["parts"][0]
        else:
            gemini_messages.append({"role": "user", "parts": [m["content"]]})
    else:
        gemini_messages.append({"role": role, "parts": [m["content"]]})

In [11]:
# See how this function creates exactly the format above

def messages(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt(website)}
    ]

In [12]:
messages(web)

[{'role': 'system',
  'content': 'You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related.'},
 {'role': 'user',
  'content': 'You are looking at a website titled BBC News - Breaking news, video and the latest top stories from the U.S. and around the world\nThe contents of this website is as follows; please provide a short summary of this website. If it includes news or announcements, then summarize these too.\n\nSkip to content\nBritish Broadcasting Corporation\nHome\nNews\nSport\nBusiness\nInnovation\nCulture\nArts\nTravel\nEarth\nAudio\nVideo\nLive\nIsrael-Gaza War\nWar in Ukraine\nUS & Canada\nUK\nAfrica\nAsia\nAustralia\nEurope\nLatin America\nMiddle East\nIn Pictures\nBBC InDepth\nBBC Verify\nHome\nNews\nIsrael-Gaza War\nWar in Ukraine\nUS & Canada\nUK\nUK Politics\nEngland\nN. Ireland\nN. Ireland Politics\nScotland\nScotland Politics\nWales\nWales Politics\nAfrica\nAsia\nChina\nIndia\nAustralia\nEur

In [13]:
def summarize(url):
    website = Website(url)
    model = genai.GenerativeModel("gemini-2.5-flash-lite")
    response = model.generate_content(message)
    return response.text

In [14]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [15]:
summarize("https://www.bbc.com/news")

"Hello there! It's wonderful to hear from you! Welcome! I'm excited to be able to communicate with you.\n\nHow can I help you today? I'm ready to chat about anything you have in mind! 😊"

In [16]:
display_summary("https://www.bbc.com/news")

Hi there! It's wonderful to hear from you! Welcome! I'm excited to be your first ever message recipient. How can I help you today? Is there anything you'd like to talk about, learn, or create? I'm ready!

In [17]:
email_content = """
Hi Nithish,
Greetings.! 
Thank you for your recent technical discussions with Zlendo Technologies Private Limited. We are delighted to offer you the position of Intern with our company, starting on 11-08-2025. Please look at the attached document in the email for complete details about your employment.
We appreciate your interest in growing with us and would like to welcome you to our family at Zlendo Technologies Private Limited. Send us a confirmation email to let us know if you've accepted the offer.

Thanks & Regards,

Zlendo HR Team
"""

gemini_messages = [
    {"role": "user", "parts": [f"Suggest a short, professional subject line for this email:\n{email_content}"]}
]

In [18]:
model = genai.GenerativeModel("gemini-2.5-flash-lite")
chat = model.start_chat(history=gemini_messages)
response = chat.send_message("Please give me only the subject line.")
print(response.text)

Job Offer: Intern at Zlendo Technologies


In [19]:
import json

In [20]:
OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL = "llama3.2"

In [21]:
def ask_ollama(prompt):
    payload = { 
        "model":MODEL,
        "messages" :[{"role":"user", "content" : prompt}],
        "stream" : False
}

    response = requests.post(OLLAMA_API , headers=HEADERS , data=json.dumps(payload))
    data = response.json()
    print("DEBUG Ollama response:", data)  # See the actual format
    
    return data["message"]["content"]

In [22]:
print(ask_ollama("Write a short subject line for an email about a brochure draft being ready for review."))

DEBUG Ollama response: {'model': 'llama3.2', 'created_at': '2025-08-11T14:08:57.6608961Z', 'message': {'role': 'assistant', 'content': 'Here are a few options:\n\n1. "New Brochure Draft Available for Review"\n2. "Review Request: Upcoming Brochure Project"\n3. "Draft of New Brochure Now Ready for Feedback"\n\nChoose the one that best fits your tone and style!'}, 'done_reason': 'stop', 'done': True, 'total_duration': 24872390200, 'load_duration': 9770084700, 'prompt_eval_count': 42, 'prompt_eval_duration': 5300674500, 'eval_count': 54, 'eval_duration': 9777946200}
Here are a few options:

1. "New Brochure Draft Available for Review"
2. "Review Request: Upcoming Brochure Project"
3. "Draft of New Brochure Now Ready for Feedback"

Choose the one that best fits your tone and style!
